In [ ]:
# sklearn has a lot of warnings.... suppress them all
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

import argparse
import copy
import glob
import json
import math
import os
import pickle
import re
import scipy
import sys
import random
import torch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats

from itertools import combinations
from numpy.random import RandomState

from collections import defaultdict
from os.path import exists
from scipy import linalg
from scipy.io import loadmat
from scipy.special import logsumexp
from scipy.stats import multivariate_normal
from scipy.spatial import distance
from sklearn import mixture
from sklearn.covariance import EmpiricalCovariance
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sys import getsizeof
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200


from sklearn.metrics import roc_auc_score
from scipy import stats
from scipy.stats import norm

sys.path.append("/orcd/data/jhm/001/om2/bjmedina/memory/utils/")

#from util_plotting import convertModelResponsesToDataFrame

sys.path.append('/orcd/data/jhm/001/om2/bjmedina/')


In [ ]:
prng = RandomState()
transform = False
z_score = False

noise_std_dev = 1
stim_filenames = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/filenames.json"
# Opening JSON file
f = open(stim_filenames)
stim_files = json.load(f)

In [ ]:
representation = np.load("/orcd/data/jhm/001/om2/bjmedina/BOLIVIA2024/assets/textures/texture_statistics_1707.npy")
data_transform = np.load("/orcd/data/jhm/001/om2/bjmedina/BOLIVIA2024/assets/textures/texture_1707_transform.npy")

clip_to_idx = {}

for j in range(len(stim_files)):
    clip_to_idx[stim_files[j]['filename']] = int(stim_files[j]['filename'].split("_")[-1].split(".")[0])


mean = np.mean(representation, axis=0)
std_dev = np.std(representation, axis=0)

# Z-score each feature
representation_z = (representation - mean) / std_dev

if transform:
    representation = representation @ data_transform
    name = "transformed data"
else:
    name = "untransformed data"

print("Z-scored data:\n", representation_z)


In [ ]:
eps = prng.normal(loc=0.0, scale=0, size=representation.shape)
eps[0]

In [ ]:
representation.shape

In [ ]:
# Compute feature standard deviations
feature_std_dev = np.std(representation, axis=0)

# Avoid division by zero
feature_std_dev[feature_std_dev == 0] = 1

# Normalize representation
noise_std_dev_scaled = noise_std_dev * feature_std_dev


In [ ]:
feature_std_dev.shape

In [ ]:
eps = prng.normal(loc=0.0, scale=noise_std_dev_scaled, size=representation.shape)
max_idx, min_idx = np.argmax(feature_std_dev), np.argmin(feature_std_dev)
plt.title(f"Distribution of scaled samples from {name}")
plt.hist(eps[:, max_idx], label="Largest values", bins=100, alpha=0.5)
plt.hist(eps[:, min_idx], label="Smallest values", bins=100, alpha=0.8)
plt.legend()

In [ ]:
plt.title(f"Distribution of scaled samples from {name}")

plt.hist(eps[:, min_idx], label="Smallest values", bins=100, alpha=0.8)
plt.legend()

In [ ]:
eps = prng.normal(loc=0.0, scale=noise_std_dev, size=representation.shape)
plt.title(f"Distribution of unscaled samples {name}")
plt.hist(eps[:, max_idx], label="Largest values", bins=100, alpha=0.5)
plt.hist(eps[:, min_idx], label="Smallest values", bins=100, alpha=0.5)
plt.legend()

In [ ]:
plt.hist(feature_std_dev, bins=100)
plt.xlabel("std of a feature")
plt.ylabel("frequency")
plt.title(f"std of features for {name}")

In [ ]:
# Compute feature standard deviations
feature_std_dev = np.std(representation_z, axis=0)

# Avoid division by zero
feature_std_dev[feature_std_dev == 0] = 1

# Normalize representation
noise_std_dev_scaled = noise_std_dev * feature_std_dev


In [ ]:
eps = prng.normal(loc=0.0, scale=noise_std_dev_scaled, size=representation.shape)
max_idx, min_idx = np.argmax(feature_std_dev), np.argmin(feature_std_dev)
plt.title(f"Distribution of scaled samples from {name}_z")
plt.hist(eps[:, max_idx], label="Largest values", bins=100, alpha=0.5)
plt.hist(eps[:, min_idx], label="Smallest values", bins=100, alpha=0.8)
plt.legend()

In [ ]:
plt.hist(feature_std_dev, bins=100)
plt.xlabel("std of a feature")
plt.ylabel("frequency")
plt.title(f"std of features for {name}")

In [ ]:
representation = np.load("/orcd/data/jhm/001/om2/bjmedina/BOLIVIA2024/assets/textures/texture_statistics_1707.npy")
data_transform = np.load("/orcd/data/jhm/001/om2/bjmedina/BOLIVIA2024/assets/textures/texture_1707_transform.npy")

clip_to_idx = {}

for j in range(len(stim_files)):
    clip_to_idx[stim_files[j]['filename']] = int(stim_files[j]['filename'].split("_")[-1].split(".")[0])

if transform:

    mean = np.mean(representation, axis=0)
    std_dev = np.std(representation, axis=0)
    
    # Z-score each feature
    representation_z = (representation - mean) / std_dev
    representation_z = representation_z @ data_transform

    print("Z-scored data:\n", representation_z)
    name = "transformed data"
else:
    
    mean = np.mean(representation, axis=0)
    std_dev = np.std(representation, axis=0)
    
    # Z-score each feature
    representation_z = (representation - mean) / std_dev
    print("Z-scored data:\n", representation_z)
    name = "untransformed data"




In [ ]:
# Compute feature standard deviations
feature_std_dev = np.std(representation_z, axis=0)

# Avoid division by zero
feature_std_dev[feature_std_dev == 0] = 1

# Normalize representation
noise_std_dev_scaled = noise_std_dev * feature_std_dev


In [ ]:
eps = prng.normal(loc=0.0, scale=noise_std_dev_scaled, size=representation.shape)
max_idx, min_idx = np.argmax(feature_std_dev), np.argmin(feature_std_dev)
plt.title(f"Distribution of scaled samples from {name}_z")
plt.hist(eps[:, max_idx], label="Largest values", bins=100, alpha=0.5)
plt.hist(eps[:, min_idx], label="Smallest values", bins=100, alpha=0.8)
plt.legend()

In [ ]:
plt.hist(feature_std_dev, bins=100)
plt.xlabel("std of a feature")
plt.ylabel("frequency")
plt.title(f"std of features for {name}")